# Question 1: datetime Fundamentals and Time Series Indexing

This question focuses on datetime handling and time series indexing using patient vital signs data.

## Setup

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import os

# Set random seed for reproducibility
np.random.seed(42)

# Set plotting style
plt.style.use("default")
sns.set_style("whitegrid")

# Create output directory
os.makedirs("output", exist_ok=True)

## Part 1.1: Load and Explore Data

**Note:** This dataset contains realistic healthcare data characteristics:
- **200 patients** with daily vital signs over 1 year
- **Missing visits**: Patients miss approximately 5% of scheduled visits (realistic!)
- **Different start dates**: Not all patients start monitoring on January 1st (some join later)
- When selecting data by date ranges, you may find that some patients don't have data for certain periods - this is expected and realistic

In [7]:
# Load patient vital signs data
patient_vitals = pd.read_csv("data/patient_vitals.csv")

print("Patient vitals shape:", patient_vitals.shape)
print("Patient vitals columns:", patient_vitals.columns.tolist())

# Display sample data
print("\nPatient vitals sample:")
display(patient_vitals.head())
print("\nData summary:")
display(patient_vitals.describe())

# Check date range and missing data patterns
print(f"\nDate range: {patient_vitals['date'].min()} to {patient_vitals['date'].max()}")
print(f"Unique patients: {patient_vitals['patient_id'].nunique()}")
print(f"Total records: {len(patient_vitals)}")
print(f"Expected records (200 patients × 365 days): {200 * 365:,}")
print(f"Missing visits: ~{200 * 365 - len(patient_vitals):,} records")

Patient vitals shape: (18250, 7)
Patient vitals columns: ['date', 'patient_id', 'temperature', 'heart_rate', 'blood_pressure_systolic', 'blood_pressure_diastolic', 'weight']

Patient vitals sample:


,date,patient_id,temperature,heart_rate,blood_pressure_systolic,blood_pressure_diastolic,weight
0,2023-01-01,P0001,98.389672,71,119,84,68.996865
1,2023-01-02,P0001,98.492046,67,117,82,67.720215
2,2023-01-03,P0001,98.790163,70,113,78,67.846825
3,2023-01-04,P0001,98.635781,74,117,82,67.693993
4,2023-01-05,P0001,98.051660,67,118,83,68.228852



Data summary:


,temperature,heart_rate,blood_pressure_systolic,blood_pressure_diastolic,weight
count,18250.000000,18250.000000,18250.000000,18250.000000,18250.000000
mean,98.660538,75.097753,118.870904,76.150904,67.293055
std,0.569494,9.329731,6.457522,5.443337,13.131365
min,96.745235,55.000000,105.000000,65.000000,40.000000
25%,98.261433,67.000000,114.000000,72.000000,58.439629
50%,98.658390,75.000000,119.000000,76.000000,67.571759
75%,99.062736,83.000000,124.000000,80.000000,77.265489
max,100.876601,93.000000,133.000000,88.000000,105.829635



Date range: 2023-01-01 to 2023-12-31
Unique patients: 50
Total records: 18250
Expected records (200 patients × 365 days): 73,000
Missing visits: ~54,750 records


## Part 1.2: datetime Operations

**TODO: Perform datetime operations**

In [8]:
# TODO: Convert date column to datetime
patient_vitals["date"] = pd.to_datetime(patient_vitals["date"])

# TODO: Set datetime column as index
patient_vitals = patient_vitals.set_index("date")

# TODO: Extract year, month, day components from datetime index
patient_vitals["year"] = patient_vitals.index.year
patient_vitals["month"] = patient_vitals.index.month
patient_vitals["day"] = patient_vitals.index.day

# TODO: Calculate time differences (e.g., days since first measurement)
# Note: Since patients start at different times, calculate days_since_start per patient
# Hint: To use groupby on the 'date' column, temporarily reset the index, then set it back
# Example: patient_vitals_reset = patient_vitals.reset_index()
#          Use groupby('patient_id')['date'].transform(lambda x: (x - x.min()).dt.days)
#          Or use groupby('patient_id').apply() to calculate days from each patient's first date
#          Then: patient_vitals = patient_vitals_reset.set_index('date')
patient_vitals = patient_vitals.reset_index()
patient_vitals["days_since_start"] = patient_vitals.groupby("patient_id")[
    "date"
].transform(lambda x: (x - x.min()).dt.days)
patient_vitals = patient_vitals.set_index("date")

# TODO: Create business day ranges for clinic visit schedules
clinic_dates = pd.bdate_range(
    start=patient_vitals.index.min(), end=patient_vitals.index.max()
)
print(f"\nClinic business days range length: {len(clinic_dates)}")

# TODO: Create date ranges with different frequencies
daily_range = pd.date_range(
    start=patient_vitals.index.min(), end=patient_vitals.index.max(), freq="D"
)  # Daily monitoring schedule
weekly_range = pd.date_range(
    start=patient_vitals.index.min(), end=patient_vitals.index.max(), freq="W-MON"
)  # Weekly lab test schedule (Mondays)
monthly_range = pd.date_range(
    start=patient_vitals.index.min(), end=patient_vitals.index.max(), freq="MS"
)  # Monthly checkup schedule

print(f"\nDaily range length: {len(daily_range)}")
print(f"Weekly range length: {len(weekly_range)}")
print(f"Monthly range length: {len(monthly_range)}")

# TODO: Use date ranges to analyze visit patterns
# Check how many patient visits occurred on clinic business days vs weekends
patient_dates_set = set(patient_vitals.index.date)
clinic_dates_set = set(clinic_dates.date)
visits_on_clinic_days = len(patient_dates_set & clinic_dates_set)
visits_on_weekends = len(patient_dates_set) - visits_on_clinic_days

print(f"\nVisits on clinic business days: {visits_on_clinic_days}")
print(f"Visits on weekends: {visits_on_weekends}")
print(f"Total unique visit dates: {len(patient_dates_set)}")

# TODO: Save results as 'output/q1_datetime_analysis.csv'
# Create a DataFrame with datetime analysis results including:
# - date (datetime index or column)
# - year, month, day (extracted from datetime)
# - days_since_start (calculated time differences)
# - patient_id
# - At least one original column (e.g., temperature, heart_rate)
# Note: When saving to CSV with index=False, you'll need to convert the index to a column first
datetime_analysis = patient_vitals.reset_index()[
    [
        "date",  # <-- include date as a column
        "patient_id",
        "year",
        "month",
        "day",
        "days_since_start",
        "temperature",
    ]
]
datetime_analysis.to_csv("output/q1_datetime_analysis.csv", index=False)



Clinic business days range length: 260

Daily range length: 365
Weekly range length: 52
Monthly range length: 12

Visits on clinic business days: 260
Visits on weekends: 105
Total unique visit dates: 365


## Part 1.3: Time Zone Handling

**TODO: Handle time zones**

In [9]:
# TODO: Create timezone-aware datetime (for multi-site clinical trials)
utc_time = pd.Timestamp.now(tz="UTC")  # Current time in UTC
eastern_time = utc_time.tz_convert("US/Eastern")  # Convert to US Eastern

# TODO: Convert between different timezones
# Create timezone-aware DataFrame from patient_vitals
# patient_vitals_tz = None  # Localize to UTC
# patient_vitals_tz_eastern = None  # Convert to Eastern time
patient_vitals_tz = patient_vitals.copy()  # Localize to UTC
patient_vitals_tz.index = patient_vitals_tz.index.tz_localize("UTC")
print("\nPatient vitals timezone-localized to UTC sample:")
display(patient_vitals_tz.head(5))

patient_vitals_tz_eastern = patient_vitals_tz.copy()  # Convert to Eastern time
patient_vitals_tz_eastern.index = patient_vitals_tz_eastern.index.tz_convert(
    "US/Eastern"
)
print("\nPatient vitals converted to US/Eastern timezone sample:")
display(patient_vitals_tz_eastern.head(5))

# TODO: Handle daylight saving time transitions
# Create datetime that spans DST transition
# Note: Using UTC avoids DST ambiguity issues - UTC has no daylight saving time
# Best practice: Store data in UTC, convert to local timezones only when needed
dst_date_utc = pd.Timestamp(
    "2023-03-12 10:00:00", tz="UTC"
)  # UTC time around US DST start
dst_time_eastern = dst_date_utc.tz_convert("US/Eastern")  # Convert UTC to Eastern

print("\nDST example:")
print("UTC time:     ", dst_date_utc)
print("Eastern time: ", dst_time_eastern)

# TODO: Document timezone operations
# Create a report string with the following sections:
# 1. Original timezone: Describe what timezone your original data was in (or if it was naive)
# 2. Localization method: Explain how you localized the data (e.g., tz_localize('UTC'))
# 3. Conversion: Describe what timezone you converted to (e.g., 'US/Eastern')
# 4. DST handling: Document any issues or observations about daylight saving time transitions
#    Note: Explain why using UTC as the base timezone avoids DST ambiguity issues
# 5. Example: Show at least one example of a datetime before and after conversion
# Minimum length: 200 words
# timezone_report = """
# TODO: Document your timezone operations:
# - What timezone was your original data in?
# - How did you localize the data?
# - What timezone did you convert to?
# - What issues did you encounter with DST? (Note: Using UTC avoids DST ambiguity)
# - Include at least one example showing a datetime before and after conversion
# - Explain why UTC is recommended as the base timezone for storing temporal data

timezone_report = """
Timezone Operations Report

1. Original Timezone
The original patient_vitals data was loaded with a naive datetime column, meaning the dates had no explicit timezone information attached. In practice, this often means the timestamps are assumed to be in the local clinic or hospital time, but pandas treats them as timezone-unaware. This can become problematic when data are combined across sites or compared to systems that do use explicit timezones.

2. Localization Method
To make the timestamps explicit and comparable, we first converted the 'date' column to a proper datetime type and set it as the DataFrame index. We then localized this naive DatetimeIndex to UTC using tz_localize('UTC'). This step does not change the actual clock values; it simply tells pandas, “these timestamps should be interpreted as UTC.”

3. Conversion
After localization, we created a second version of the data by converting from UTC to the 'US/Eastern' timezone using tz_convert('US/Eastern'). This produces timezone-aware timestamps that reflect Eastern time while still representing the same underlying instants in time as the UTC values.

4. DST Handling
We illustrated daylight saving time handling with the example 2023-03-12 10:00:00 UTC, which converts to 2023-03-12 06:00:00-04:00 in US/Eastern. On this date, the Eastern timezone switches from standard time to daylight saving time, and the offset changes from UTC-5 to UTC-4. Because we store the data in UTC and only convert to local timezones when needed, pandas can automatically handle this offset change without ambiguity. There is no “missing hour” in UTC, so the representation is stable and consistent.

5. Example of Conversion
For example, 2023-03-12 10:00:00+00:00 in UTC becomes 2023-03-12 06:00:00-04:00 in US/Eastern. The underlying moment in time is identical, but the local clock time and UTC offset differ. This demonstrates why UTC is recommended as the base timezone for storing temporal data in clinical and multi-site studies: it avoids DST confusion, makes cross-site comparisons easier, and defers any local-time formatting to the very end of the analysis or reporting pipeline.
""".strip()

# TODO: Save results as 'output/q1_timezone_report.txt'
with open("output/q1_timezone_report.txt", "w") as f:
    f.write(timezone_report)


Patient vitals timezone-localized to UTC sample:


,patient_id,temperature,heart_rate,blood_pressure_systolic,blood_pressure_diastolic,weight,year,month,day,days_since_start
date,,,,,,,,,,
2023-01-01 00:00:00+00:00,P0001,98.389672,71,119,84,68.996865,2023,1,1,0
2023-01-02 00:00:00+00:00,P0001,98.492046,67,117,82,67.720215,2023,1,2,1
2023-01-03 00:00:00+00:00,P0001,98.790163,70,113,78,67.846825,2023,1,3,2
2023-01-04 00:00:00+00:00,P0001,98.635781,74,117,82,67.693993,2023,1,4,3
2023-01-05 00:00:00+00:00,P0001,98.051660,67,118,83,68.228852,2023,1,5,4



Patient vitals converted to US/Eastern timezone sample:


,patient_id,temperature,heart_rate,blood_pressure_systolic,blood_pressure_diastolic,weight,year,month,day,days_since_start
date,,,,,,,,,,
2022-12-31 19:00:00-05:00,P0001,98.389672,71,119,84,68.996865,2023,1,1,0
2023-01-01 19:00:00-05:00,P0001,98.492046,67,117,82,67.720215,2023,1,2,1
2023-01-02 19:00:00-05:00,P0001,98.790163,70,113,78,67.846825,2023,1,3,2
2023-01-03 19:00:00-05:00,P0001,98.635781,74,117,82,67.693993,2023,1,4,3
2023-01-04 19:00:00-05:00,P0001,98.051660,67,118,83,68.228852,2023,1,5,4



DST example:
UTC time:      2023-03-12 10:00:00+00:00
Eastern time:  2023-03-12 06:00:00-04:00


## Submission Checklist

Before moving to Question 2, verify you've created:

- [ ] `output/q1_datetime_analysis.csv` - datetime analysis results
- [ ] `output/q1_timezone_report.txt` - timezone handling report
